# bias-kb 公開抜粋 — offline 実行単位を再実行する / Re-run the bundled execution units

このノートブックは `rerun.py` と同じことをします: 各 offline 実行単位の凍結入力の sha256 を照合し、`entry` コマンドを実行して、記録された期待値 `expected` と比較します。
結果は Atlas の接地タブ（https://jxta.github.io/bias-kb-atlas/#v=grounding ）や `kb/rerun-latest.json`（CI の最終再実行）と同じ形です。

This notebook does what `rerun.py` does: for each offline execution unit it checks the sha256 of the frozen inputs, runs the `entry` command and compares the output with the recorded `expected` value. Binder: the environment has Python and sympy only; nothing is downloaded.


In [ ]:
import json, subprocess, sys, os, hashlib, time
ROOT = os.getcwd()
data = json.load(open('kb/nodes.json', encoding='utf-8'))
meta, nodes = data['meta'], data['nodes']
print('record layer', meta['stats']['full_n'], 'nodes /', 'excerpt', meta['stats']['n'], '/ generated', meta['generated_at'], '/ ref', meta['source_ref'])
units = [n for n in nodes if n['type'] == 'ExecutionUnit' and (n.get('env') or {}).get('offline') and not n.get('_stub')]
print(len(units), 'offline execution units:', ', '.join(u['id'] for u in units))

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

rows = []
for u in units:
    inputs = [i for i in (u.get('inputs') or []) if isinstance(i, dict) and i.get('ref')]
    missing = [i['ref'] for i in inputs if str(i['ref']).startswith('annex') or not os.path.exists(i['ref'])]
    if missing:
        rows.append((u['id'], u.get('tier'), 'NOT-BUNDLED', '', '', ', '.join(missing))); continue
    sha_ok = all((not i.get('sha256')) or sha256(i['ref']) == i['sha256'] for i in inputs)
    t0 = time.time()
    r = subprocess.run(u['entry'], shell=True, capture_output=True, text=True, cwd=ROOT, timeout=600)
    got = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else ''
    want = str((u.get('expected') or [{}])[0].get('value'))
    status = 'PASS' if sha_ok and (got == want or want in got or got in want) else ('SHA-MISMATCH' if not sha_ok else 'MISMATCH')
    rows.append((u['id'], u.get('tier'), status, got[:60], want[:60], f'{time.time()-t0:.2f}s'))

w = max(len(r[0]) for r in rows)
print(f"{'unit':<{w}}  tier    status        got / want")
for r in rows:
    print(f"{r[0]:<{w}}  {str(r[1]):<7} {r[2]:<13} {r[3] + ' / ' + r[4] if r[2] != 'NOT-BUNDLED' else '(' + r[5] + ')'}")
ran = [r for r in rows if r[2] != 'NOT-BUNDLED']
print(f"\n{sum(1 for r in ran if r[2]=='PASS')}/{len(ran)} PASS ({sum(1 for r in rows if r[2]=='NOT-BUNDLED')} not bundled)")

## 主張 → 証拠 → 実行単位 の鎖を辿る / Walk a grounding chain

`C-1q-law`（1/q 法則）を例に、主張が指す証拠と、証拠を再実行で確かめる実行単位を機械的に辿ります（Atlas の接地タブの 1 行と同じ）。


In [ ]:
byid = {n['id']: n for n in nodes}
k4 = meta.get('k4') or {}
def chain(cid):
    c = byid[cid]; print(c['id'], '—', c['label'], f"[{c.get('status')}]")
    for ev in c.get('supported_by', []) + c.get('refuted_by', []):
        E = byid.get(ev)
        if not E or E.get('_stub'): print('  evidence', ev, '(private stub)' if E and E.get('_stub') else ''); continue
        xs = list(dict.fromkeys(E.get('grounded_in', []) + [i['from'] for i in E['_in'] if i['rel'] == 'verifies']))
        print('  evidence', ev, '—', E['label'][:60])
        for x in xs:
            X = byid.get(x)
            if not X: continue
            r = k4.get(x) or {}
            print('    execution', x, 'tier', X.get('tier'), 'offline' if (X.get('env') or {}).get('offline') else 'full-run', '| k4', r.get('status', '—'), (r.get('got') or '')[:40])
chain('C-1q-law')
print()
chain('C-d19-sign-reversal')

各ノードは JS なしの単体ページ `n/<id>.html` と JSON-LD `n/<id>.json` でも読めます。機械可読の入口は `llms.txt`、束全体の記述は `ro-crate-metadata.json`。
